In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

## 例1

### 方法一

In [2]:
# from langchain import hub

# rephrase_prompt = hub.pull("langchain-ai/chat-langchain-rephrase")

In [3]:
# rephrase_prompt

```
PromptTemplate(input_variables=['chat_history', 'input'], metadata={'lc_hub_owner': 'langchain-ai', 'lc_hub_repo': 'chat-langchain-rephrase', 'lc_hub_commit_hash': 'fb7ddb56be11b2ab10d176174dae36faa2a9a6ba13187c8b2b98315f6ca7d136'}, template='Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question.\n\nChat History:\n{chat_history}\nFollow Up Input: {input}\nStandalone Question:')
```

In [4]:
# print(rephrase_prompt.template)

```
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question.

Chat History:
{chat_history}
Follow Up Input: {input}
Standalone Question:
```

In [5]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

rephrase_system_prompt = """根据以下对话和后续问题，重新表述后续问题为一个独立的问题。

聊天记录：
{chat_history}
后续输入：{input}
独立问题："""

rephrase_prompt = PromptTemplate.from_template(rephrase_system_prompt)

In [6]:
rephrase_prompt

PromptTemplate(input_variables=['chat_history', 'input'], template='根据以下对话和后续问题，重新表述后续问题为一个独立的问题。\n\n聊天记录：\n{chat_history}\n后续输入：{input}\n独立问题：')

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
output_parser = StrOutputParser()

rephrase_chain = rephrase_prompt | llm | output_parser

#### test 1 - chat_history 为空

In [8]:
chat_history = []

In [9]:
query = "气候变化会对哪些领域产生影响"

In [10]:
rephrase_prompt.invoke({"input": query, "chat_history": chat_history})

StringPromptValue(text='根据以下对话和后续问题，重新表述后续问题为一个独立的问题。\n\n聊天记录：\n[]\n后续输入：气候变化会对哪些领域产生影响\n独立问题：')

In [11]:
rephrase_query = rephrase_chain.invoke({"input": query, "chat_history": chat_history})
rephrase_query

'气候变化对哪些领域产生影响？'

#### test 2 - chat_history 不为空

In [12]:
from langchain_core.messages import AIMessage, HumanMessage

chat_history = [
    HumanMessage(content="气候变化会对哪些领域产生影响"),
    AIMessage(content="农业、水资源、生态系统、城市发展、健康等"),
]

chat_history

[HumanMessage(content='气候变化会对哪些领域产生影响'),
 AIMessage(content='农业、水资源、生态系统、城市发展、健康等')]

In [13]:
query = "详细讲下对农业的影响"

In [14]:
rephrase_prompt.invoke({"input": query, "chat_history": chat_history})

StringPromptValue(text="根据以下对话和后续问题，重新表述后续问题为一个独立的问题。\n\n聊天记录：\n[HumanMessage(content='气候变化会对哪些领域产生影响'), AIMessage(content='农业、水资源、生态系统、城市发展、健康等')]\n后续输入：详细讲下对农业的影响\n独立问题：")

In [15]:
rephrase_query = rephrase_chain.invoke({"input": query, "chat_history": chat_history})
rephrase_query

'气候变化对农业产生哪些影响？'

### 方法二

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder

rephrase_system_prompt = (
    "根据聊天记录和最新的用户问题，该问题可能涉及到聊天记录中的内容，"
    "请形成一个能独立理解而不需要聊天记录的单独问题。"
    "不要回答最新的用户问题，只是重新表述最新的用户问题或者原样返回最新的用户问题。"
)

rephrase_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rephrase_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [17]:
rephrase_prompt

ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='根据聊天记录和最新的用户问题，该问题可能涉及到聊天记录中的内容，请形成一个能独立理解而不需要聊天记录的单独问题。不要回答最新的用户问题，只是重新表述最新的用户问题或者原样返回最新的用户问题。')), MessagesPlaceholder(variable_name='chat_history'), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])

In [18]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI() # model="gpt-4o", temperature=0
output_parser = StrOutputParser()

rephrase_chain = rephrase_prompt | llm | output_parser

#### test 1 - chat_history 为空

In [19]:
chat_history = []

In [20]:
query = "气候变化会对哪些领域产生影响"

In [21]:
rephrase_prompt.invoke({"input": query, "chat_history": chat_history})

ChatPromptValue(messages=[SystemMessage(content='根据聊天记录和最新的用户问题，该问题可能涉及到聊天记录中的内容，请形成一个能独立理解而不需要聊天记录的单独问题。不要回答最新的用户问题，只是重新表述最新的用户问题或者原样返回最新的用户问题。'), HumanMessage(content='气候变化会对哪些领域产生影响')])

In [22]:
rephrase_query = rephrase_chain.invoke({"input": query, "chat_history": chat_history})
rephrase_query

'气候变化会对哪些领域产生影响？'

In [23]:
rephrase_prompt.invoke({"input": query, "chat_history": chat_history})

ChatPromptValue(messages=[SystemMessage(content='根据聊天记录和最新的用户问题，该问题可能涉及到聊天记录中的内容，请形成一个能独立理解而不需要聊天记录的单独问题。不要回答最新的用户问题，只是重新表述最新的用户问题或者原样返回最新的用户问题。'), HumanMessage(content='气候变化会对哪些领域产生影响')])

#### test 2 - chat_history 不为空

In [24]:
from langchain_core.messages import AIMessage, HumanMessage

chat_history = [
    HumanMessage(content="气候变化会对哪些领域产生影响"),
    AIMessage(content="农业、水资源、生态系统、城市发展、健康等"),
]

chat_history

[HumanMessage(content='气候变化会对哪些领域产生影响'),
 AIMessage(content='农业、水资源、生态系统、城市发展、健康等')]

In [25]:
query = "详细讲下对农业的影响"

In [26]:
rephrase_prompt.invoke({"input": query, "chat_history": chat_history})

ChatPromptValue(messages=[SystemMessage(content='根据聊天记录和最新的用户问题，该问题可能涉及到聊天记录中的内容，请形成一个能独立理解而不需要聊天记录的单独问题。不要回答最新的用户问题，只是重新表述最新的用户问题或者原样返回最新的用户问题。'), HumanMessage(content='气候变化会对哪些领域产生影响'), AIMessage(content='农业、水资源、生态系统、城市发展、健康等'), HumanMessage(content='详细讲下对农业的影响')])

In [27]:
rephrase_query = rephrase_chain.invoke({"input": query, "chat_history": chat_history})
rephrase_query

'气候变化对农业的影响主要包括以下几个方面：\n1. 气温升高：气候变暖会导致气温升高，可能会影响作物的生长季节和生长周期，影响作物的产量和质量。\n2. 降水变化：气候变化会引起降水模式的改变，可能导致干旱或洪涝等极端天气事件，影响农作物的生长和灌溉水资源的供应。\n3. 病虫害扩散：气候变化可能会导致病虫害的扩散范围扩大，对作物生长造成威胁，增加农业生产的风险。\n4. 土壤质量变化：气候变化会影响土壤中的养分含量和水分分布，可能导致土壤质量下降，影响农作物的生长。\n5. 生态系统变化：气候变化还可能影响农业生态系统，如影响农作物和农作物间的生态平衡，进而影响农业的可持续发展。\n\n总的来说，气候变化对农业产生的影响是复杂而多样的，需要农业领域的相关研究和政策支持来应对这些挑战。'

#### test 3 - chat_history 换种方式表示

In [28]:
chat_history = [
    ("human", "气候变化会对哪些领域产生影响"),
    ("ai", "农业、水资源、生态系统、城市发展、健康等"),
]

chat_history

[('human', '气候变化会对哪些领域产生影响'), ('ai', '农业、水资源、生态系统、城市发展、健康等')]

In [29]:
query = "详细讲下对农业的影响"

In [30]:
rephrase_query = rephrase_chain.invoke({"input": query, "chat_history": chat_history})
rephrase_query

'气候变化对农业产生了多方面的影响。一方面，气候变化导致了气温升高、干旱频发、极端天气增多等现象，这些因素使得农作物生长受到影响，产量下降。另一方面，气候变化也可能引发疾病虫害的扩散，对农作物产生威胁。同时，降水不均、气候变暖也会改变农作物的生长季节和地理分布，需要调整种植方式和时间。\n\n另外，气候变化还会对农业水资源管理产生影响，降水模式的改变可能导致农田灌溉水源减少，需要采取更加有效的水资源管理措施。此外，气候变化还可能影响农作物的品质和农产品的价格，对农业经济产生影响。因此，农业领域需要适应气候变化的挑战，采取相应的调整措施以保障粮食生产和农业可持续发展。'

In [31]:
rephrase_prompt.invoke({"input": query, "chat_history": chat_history})

ChatPromptValue(messages=[SystemMessage(content='根据聊天记录和最新的用户问题，该问题可能涉及到聊天记录中的内容，请形成一个能独立理解而不需要聊天记录的单独问题。不要回答最新的用户问题，只是重新表述最新的用户问题或者原样返回最新的用户问题。'), HumanMessage(content='气候变化会对哪些领域产生影响'), AIMessage(content='农业、水资源、生态系统、城市发展、健康等'), HumanMessage(content='详细讲下对农业的影响')])

## 例2

### 第一次尝试

#### 创建链

In [40]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Prompt
rephrase_system_prompt = (
    "根据聊天记录和最新的用户问题，该问题可能涉及到聊天记录中的内容，"
    "请形成一个能独立理解而不需要聊天记录的单独问题。"
    "不要回答这个问题，只是重新表述或者原样返回。"
)

rephrase_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rephrase_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# LLM
llm = ChatOpenAI()

# Output Parser
output_parser = StrOutputParser()

# Chain
rephrase_chain = rephrase_prompt | llm | output_parser

In [41]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rephrase_chain = RunnableWithMessageHistory(
    rephrase_chain, 
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [42]:
store

{}

#### 运行链

In [39]:
from langchain.globals import set_debug

set_debug(True)

In [43]:
query = "气候变化会对哪些领域产生影响"

In [44]:
config={"configurable": {"session_id": "1"}}

In [45]:
conversational_rephrase_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "气候变化会对哪些领域产生影响"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "气候变化会对哪些领域产生影响"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "气候变化会对哪些领域产生影响"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "气候变化会对哪些领域产生影响"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
{
  "output": []
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
{
  "chat_history": []
}
[chain/end] [chain:RunnableWithMe

'气候变化将对哪些领域产生影响？'

In [46]:
store

{'1': InMemoryChatMessageHistory(messages=[HumanMessage(content='气候变化会对哪些领域产生影响'), AIMessage(content='气候变化将对哪些领域产生影响？')])}

### 第二次尝试

In [47]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Prompt
rephrase_system_prompt = (
    "根据聊天记录和最新的用户问题，该问题可能涉及到聊天记录中的内容，"
    "请形成一个能独立理解而不需要聊天记录的单独问题。"
    "不要回答这个问题，只是重新表述或者原样返回。"
)

rephrase_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rephrase_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# LLM
llm = ChatOpenAI()

# Output Parser
output_parser = StrOutputParser()

# Chain
rephrase_chain = rephrase_prompt | llm | output_parser | llm | output_parser

In [48]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rephrase_chain = RunnableWithMessageHistory(
    rephrase_chain, 
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [49]:
store

{}

#### 运行链

In [50]:
query = "气候变化会对哪些领域产生影响"

In [51]:
config={"configurable": {"session_id": "1"}}

In [52]:
conversational_rephrase_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "气候变化会对哪些领域产生影响"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "气候变化会对哪些领域产生影响"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "气候变化会对哪些领域产生影响"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "气候变化会对哪些领域产生影响"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
{
  "output": []
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
{
  "chat_history": []
}
[chain/end] [chain:RunnableWithMe

'气候变化对以下领域产生影响：\n\n1. 农业：气候变化可能导致降雨量和气温的变化，影响农作物的生长和产量。\n\n2. 水资源：气候变化可能导致降雨模式的变化，影响水资源的供应和分布。\n\n3. 自然灾害：气候变化可能导致极端天气事件的增加，如暴雨、洪水、干旱、飓风等，增加自然灾害的风险。\n\n4. 生态系统：气候变化可能导致生态系统的变化，影响物种的分布和生存。\n\n5. 健康：气候变化可能导致疾病的传播范围扩大，影响人类健康。\n\n6. 经济：气候变化可能对不同行业和市场产生影响，如农业、旅游业、保险业等。\n\n7. 社会稳定：气候变化可能导致资源紧缺和环境恶化，影响社会稳定和安全。'

In [53]:
store

{'1': InMemoryChatMessageHistory(messages=[HumanMessage(content='气候变化会对哪些领域产生影响'), AIMessage(content='气候变化对以下领域产生影响：\n\n1. 农业：气候变化可能导致降雨量和气温的变化，影响农作物的生长和产量。\n\n2. 水资源：气候变化可能导致降雨模式的变化，影响水资源的供应和分布。\n\n3. 自然灾害：气候变化可能导致极端天气事件的增加，如暴雨、洪水、干旱、飓风等，增加自然灾害的风险。\n\n4. 生态系统：气候变化可能导致生态系统的变化，影响物种的分布和生存。\n\n5. 健康：气候变化可能导致疾病的传播范围扩大，影响人类健康。\n\n6. 经济：气候变化可能对不同行业和市场产生影响，如农业、旅游业、保险业等。\n\n7. 社会稳定：气候变化可能导致资源紧缺和环境恶化，影响社会稳定和安全。')])}

In [54]:
query = "详细讲下对农业的影响"

In [55]:
conversational_rephrase_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "详细讲下对农业的影响"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "详细讲下对农业的影响"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "详细讲下对农业的影响"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "详细讲下对农业的影响"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history] s] 

'\n\n7. 生态系统平衡：气候变化会影响生态系统的平衡，可能导致一些农作物的生长环境发生变化，使得原本适宜种植的作物在某些地区变得困难。\n\n8. 农业生产周期：气候变化可能导致农作物的生产周期发生变化，从而影响农业生产的计划和效率。\n\n9. 资源管理：气候变化可能导致资源的供应和需求发生变化，农业生产所需的水、肥料、种子等资源可能会受到影响。\n\n10. 经济影响：气候变化对农业的影响还可能对农民的经济状况产生影响，因为产量的变化可能会影响作物的市场价格，从而影响农民的收入。\n\n因此，农业部门需要不断的进行气候变化风险评估，采取相应的措施来适应气候变化的影响，保障农业生产的可持续性和稳定性。同时，还需要加强科研力量，研发适应气候变化的新品种和技术，提高农业抗灾能力和生产效率。'

In [56]:
store

{'1': InMemoryChatMessageHistory(messages=[HumanMessage(content='气候变化会对哪些领域产生影响'), AIMessage(content='气候变化对以下领域产生影响：\n\n1. 农业：气候变化可能导致降雨量和气温的变化，影响农作物的生长和产量。\n\n2. 水资源：气候变化可能导致降雨模式的变化，影响水资源的供应和分布。\n\n3. 自然灾害：气候变化可能导致极端天气事件的增加，如暴雨、洪水、干旱、飓风等，增加自然灾害的风险。\n\n4. 生态系统：气候变化可能导致生态系统的变化，影响物种的分布和生存。\n\n5. 健康：气候变化可能导致疾病的传播范围扩大，影响人类健康。\n\n6. 经济：气候变化可能对不同行业和市场产生影响，如农业、旅游业、保险业等。\n\n7. 社会稳定：气候变化可能导致资源紧缺和环境恶化，影响社会稳定和安全。'), HumanMessage(content='详细讲下对农业的影响'), AIMessage(content='\n\n7. 生态系统平衡：气候变化会影响生态系统的平衡，可能导致一些农作物的生长环境发生变化，使得原本适宜种植的作物在某些地区变得困难。\n\n8. 农业生产周期：气候变化可能导致农作物的生产周期发生变化，从而影响农业生产的计划和效率。\n\n9. 资源管理：气候变化可能导致资源的供应和需求发生变化，农业生产所需的水、肥料、种子等资源可能会受到影响。\n\n10. 经济影响：气候变化对农业的影响还可能对农民的经济状况产生影响，因为产量的变化可能会影响作物的市场价格，从而影响农民的收入。\n\n因此，农业部门需要不断的进行气候变化风险评估，采取相应的措施来适应气候变化的影响，保障农业生产的可持续性和稳定性。同时，还需要加强科研力量，研发适应气候变化的新品种和技术，提高农业抗灾能力和生产效率。')])}

## 例3

### 第一次尝试

#### 创建链

In [58]:
from dotenv import load_dotenv
from langchain import hub
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# Load
file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

# Split
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

# Embed
embedding = OpenAIEmbeddings()

# Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

# Retrieve
retriever = vectorstore.as_retriever()

In [65]:
# Generate

# Prompt
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# LLM
llm = ChatOpenAI()

# Output Parser
output_parser = StrOutputParser()

# Chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": (lambda x: x["input"]) | retriever | format_docs, "input": (lambda x: x["input"])}
    | prompt
    | llm
    | output_parser
)

In [81]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain, 
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [82]:
store

{}

#### 运行链

In [83]:
query = "什么是提示工程？"

In [84]:
config={"configurable": {"session_id": "2"}}

In [85]:
conversational_rag_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
{
  "output": []
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
{
  "chat_history": []
}
[chain/end] [chain:RunnableWithMessageHistory > chain:ins

'提示工程是指如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期的结果。这是一门基于实验的科学，旨在实现模型的对齐和可控制性，需要大量的试验和启发式方法。'

In [86]:
store

{'2': InMemoryChatMessageHistory(messages=[HumanMessage(content='什么是提示工程？'), AIMessage(content='提示工程是指如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期的结果。这是一门基于实验的科学，旨在实现模型的对齐和可控制性，需要大量的试验和启发式方法。')])}

In [87]:
query = "它都有哪些技术？"

In [88]:
conversational_rag_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history] s] Exiting 

'这段文本提到了提示工程中的技术，包括提示格式、训练示例的选择以及它们的顺序，这些因素对模型性能具有重要影响。另外，还涉及少样本分类的研究，以及GPT-3模型存在的偏见问题和如何克服这些偏见的方法。'

In [89]:
store

{'2': InMemoryChatMessageHistory(messages=[HumanMessage(content='什么是提示工程？'), AIMessage(content='提示工程是指如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期的结果。这是一门基于实验的科学，旨在实现模型的对齐和可控制性，需要大量的试验和启发式方法。'), HumanMessage(content='它都有哪些技术？'), AIMessage(content='这段文本提到了提示工程中的技术，包括提示格式、训练示例的选择以及它们的顺序，这些因素对模型性能具有重要影响。另外，还涉及少样本分类的研究，以及GPT-3模型存在的偏见问题和如何克服这些偏见的方法。')])}

In [90]:
query = "我刚刚说了什么？"

In [91]:
conversational_rag_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history] s] Exiting 

'你刚刚说你敢打赌视频游戏比电影更有趣。'

In [92]:
store

{'2': InMemoryChatMessageHistory(messages=[HumanMessage(content='什么是提示工程？'), AIMessage(content='提示工程是指如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期的结果。这是一门基于实验的科学，旨在实现模型的对齐和可控制性，需要大量的试验和启发式方法。'), HumanMessage(content='它都有哪些技术？'), AIMessage(content='这段文本提到了提示工程中的技术，包括提示格式、训练示例的选择以及它们的顺序，这些因素对模型性能具有重要影响。另外，还涉及少样本分类的研究，以及GPT-3模型存在的偏见问题和如何克服这些偏见的方法。'), HumanMessage(content='我刚刚说了什么？'), AIMessage(content='你刚刚说你敢打赌视频游戏比电影更有趣。')])}

### 第二次尝试

#### 创建链

In [93]:
from dotenv import load_dotenv
from langchain import hub
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# Load
file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

# Split
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

# Embed
embedding = OpenAIEmbeddings()

# Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

# Retrieve
retriever = vectorstore.as_retriever()

In [94]:
from langchain_core.prompts import MessagesPlaceholder

# Generate

# Prompt
# prompt = hub.pull("rlm/rag-prompt")
rag_system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rag_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# LLM
llm = ChatOpenAI()

# Output Parser
output_parser = StrOutputParser()

# Chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": (lambda x: x["input"]) | retriever | format_docs, 
        "input": (lambda x: x["input"]),
        "chat_history": (lambda x: x["chat_history"])
    }
    | rag_prompt
    | llm
    | output_parser
)

In [95]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain, 
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [96]:
store

{}

#### 运行链

In [97]:
query = "什么是提示工程？"

In [98]:
config={"configurable": {"session_id": "3"}}

In [99]:
conversational_rag_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
{
  "output": []
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
{
  "chat_history": []
}
[chain/end] [chain:RunnableWithMessageHistory > chain:ins

'提示工程是指如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期结果的实验科学。它旨在实现模型的对齐和可控制性，需要大量试验和启发式方法。'

In [100]:
store

{'3': InMemoryChatMessageHistory(messages=[HumanMessage(content='什么是提示工程？'), AIMessage(content='提示工程是指如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期结果的实验科学。它旨在实现模型的对齐和可控制性，需要大量试验和启发式方法。')])}

In [101]:
query = "它都有哪些技术？"

In [102]:
conversational_rag_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history] s] Exiting 

'提示工程涉及的技术包括提示设计、示例选择、示例顺序、标签偏见纠正等。这些技术旨在最大化模型性能，并解决模型可能存在的偏见和变异性问题。'

In [103]:
store

{'3': InMemoryChatMessageHistory(messages=[HumanMessage(content='什么是提示工程？'), AIMessage(content='提示工程是指如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期结果的实验科学。它旨在实现模型的对齐和可控制性，需要大量试验和启发式方法。'), HumanMessage(content='它都有哪些技术？'), AIMessage(content='提示工程涉及的技术包括提示设计、示例选择、示例顺序、标签偏见纠正等。这些技术旨在最大化模型性能，并解决模型可能存在的偏见和变异性问题。')])}

In [104]:
query = "我刚刚说了什么？"

In [105]:
conversational_rag_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history] s] Exiting 

'您刚刚问了关于提示工程的技术有哪些的问题。'

In [106]:
store

{'3': InMemoryChatMessageHistory(messages=[HumanMessage(content='什么是提示工程？'), AIMessage(content='提示工程是指如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期结果的实验科学。它旨在实现模型的对齐和可控制性，需要大量试验和启发式方法。'), HumanMessage(content='它都有哪些技术？'), AIMessage(content='提示工程涉及的技术包括提示设计、示例选择、示例顺序、标签偏见纠正等。这些技术旨在最大化模型性能，并解决模型可能存在的偏见和变异性问题。'), HumanMessage(content='我刚刚说了什么？'), AIMessage(content='您刚刚问了关于提示工程的技术有哪些的问题。')])}

### 第三次尝试

#### 创建链

In [107]:
from dotenv import load_dotenv
from langchain import hub
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# Load
file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

# Split
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

# Embed
embedding = OpenAIEmbeddings()

# Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

# Retrieve
retriever = vectorstore.as_retriever()

In [108]:
from langchain_core.prompts import MessagesPlaceholder

# Generate

# Prompt
# prompt = hub.pull("rlm/rag-prompt")
rag_system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rag_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# LLM
llm = ChatOpenAI()

# Output Parser
output_parser = StrOutputParser()

In [109]:
rephrase_system_prompt = (
    "根据聊天记录和最新的用户问题，该问题可能涉及到聊天记录中的内容，"
    "请形成一个能独立理解而不需要聊天记录的单独问题。"
    "不要回答这个问题，只是重新表述或者原样返回。"
)

rephrase_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rephrase_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [126]:
# Chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": rephrase_prompt | llm | output_parser | retriever | format_docs, 
        "input": (lambda x: x["input"]),
        "chat_history": (lambda x: x["chat_history"]),
    }
    | rag_prompt
    | llm
    | output_parser
)

In [127]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain, 
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [128]:
store

{}

#### 运行链

In [129]:
query = "什么是提示工程？"

In [130]:
config={"configurable": {"session_id": "4"}}

In [131]:
conversational_rag_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "什么是提示工程？"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
{
  "output": []
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
{
  "chat_history": []
}
[chain/end] [chain:RunnableWithMessageHistory > chain:ins

'提示工程涉及如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期结果。这是一门基于实验的科学，旨在实现模型的对齐和可控制性。'

In [132]:
store

{'4': InMemoryChatMessageHistory(messages=[HumanMessage(content='什么是提示工程？'), AIMessage(content='提示工程涉及如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期结果。这是一门基于实验的科学，旨在实现模型的对齐和可控制性。')])}

In [133]:
query = "它都有哪些技术？"

In [134]:
conversational_rag_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "它都有哪些技术？"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history] s] Exiting 

'提示工程涉及一系列技术，包括设计有效的提示文本、调整文本长度、选择合适的文本格式和结构等。另外，还包括对模型输出进行后处理以满足特定需求，以及通过启发式方法和实验来优化提示工程的效果。'

In [135]:
store

{'4': InMemoryChatMessageHistory(messages=[HumanMessage(content='什么是提示工程？'), AIMessage(content='提示工程涉及如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期结果。这是一门基于实验的科学，旨在实现模型的对齐和可控制性。'), HumanMessage(content='它都有哪些技术？'), AIMessage(content='提示工程涉及一系列技术，包括设计有效的提示文本、调整文本长度、选择合适的文本格式和结构等。另外，还包括对模型输出进行后处理以满足特定需求，以及通过启发式方法和实验来优化提示工程的效果。')])}

In [136]:
query = "我刚刚说了什么？"

In [137]:
conversational_rag_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "我刚刚说了什么？"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history] s] Exiting 

'你问了关于提示工程的技术有哪些的问题。提示工程涉及一系列技术，包括设计有效的提示文本、调整文本长度、选择合适的文本格式和结构等。另外，还包括对模型输出进行后处理以满足特定需求，以及通过启发式方法和实验来优化提示工程的效果。'

In [138]:
store

{'4': InMemoryChatMessageHistory(messages=[HumanMessage(content='什么是提示工程？'), AIMessage(content='提示工程涉及如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期结果。这是一门基于实验的科学，旨在实现模型的对齐和可控制性。'), HumanMessage(content='它都有哪些技术？'), AIMessage(content='提示工程涉及一系列技术，包括设计有效的提示文本、调整文本长度、选择合适的文本格式和结构等。另外，还包括对模型输出进行后处理以满足特定需求，以及通过启发式方法和实验来优化提示工程的效果。'), HumanMessage(content='我刚刚说了什么？'), AIMessage(content='你问了关于提示工程的技术有哪些的问题。提示工程涉及一系列技术，包括设计有效的提示文本、调整文本长度、选择合适的文本格式和结构等。另外，还包括对模型输出进行后处理以满足特定需求，以及通过启发式方法和实验来优化提示工程的效果。')])}

In [139]:
query = "详细介绍下少样本提示技术"

In [140]:
conversational_rag_chain.invoke(
    {"input": query},
    config=config,
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "详细介绍下少样本提示技术"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "详细介绍下少样本提示技术"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] Entering Chain run with input:
{
  "input": "详细介绍下少样本提示技术"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] Entering Chain run with input:
{
  "input": "详细介绍下少样本提示技术"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history> > chain:load_history] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<chat_history>] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_hist

'少样本提示技术是一种通过在模型训练中使用少量样本来提高性能的方法。它可以通过在样本中引入新领域、语义相似性、逻辑关系等方式来进行。这种方法有助于模型在具有挑战性的任务上表现良好，并提高泛化能力。'

In [141]:
store

{'4': InMemoryChatMessageHistory(messages=[HumanMessage(content='什么是提示工程？'), AIMessage(content='提示工程涉及如何与大语言模型进行交流，通过不更新模型权重的方式引导其行为，以达到预期结果。这是一门基于实验的科学，旨在实现模型的对齐和可控制性。'), HumanMessage(content='它都有哪些技术？'), AIMessage(content='提示工程涉及一系列技术，包括设计有效的提示文本、调整文本长度、选择合适的文本格式和结构等。另外，还包括对模型输出进行后处理以满足特定需求，以及通过启发式方法和实验来优化提示工程的效果。'), HumanMessage(content='我刚刚说了什么？'), AIMessage(content='你问了关于提示工程的技术有哪些的问题。提示工程涉及一系列技术，包括设计有效的提示文本、调整文本长度、选择合适的文本格式和结构等。另外，还包括对模型输出进行后处理以满足特定需求，以及通过启发式方法和实验来优化提示工程的效果。'), HumanMessage(content='详细介绍下少样本提示技术'), AIMessage(content='少样本提示技术是一种通过在模型训练中使用少量样本来提高性能的方法。它可以通过在样本中引入新领域、语义相似性、逻辑关系等方式来进行。这种方法有助于模型在具有挑战性的任务上表现良好，并提高泛化能力。')])}